In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
import os

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <61623A3D-DA3C-3AAD-B2F0-D363151DDB3F> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so
  Expected in:     <CC2A0259-414A-3562-95F8-DB0DE0A75BD7> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib
  warn(f"Failed to load image Python extension: {e}")


In [2]:
class EmotionDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        wav_data = self.df.iloc[idx]["wav_file"]  
        valence = self.df.iloc[idx]["Valence"]
        arousal = self.df.iloc[idx]["Arousal"]
        dominance = self.df.iloc[idx]["Dominance"]
        
        
        inputs = self.processor(wav_data, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs['labels'] = torch.tensor([valence, arousal, dominance], dtype=torch.float32)
        
        return inputs


In [3]:
class Wav2Vec2ForEmotionRegression(nn.Module):
    def __init__(self):
        super(Wav2Vec2ForEmotionRegression, self).__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        #self.regression_layer = nn.Linear(self.wav2vec2.config.hidden_size, 3)  # 3 for valence, arousal, dominance
        self.rnn = nn.LSTM(input_size=self.wav2vec2.config.hidden_size, hidden_size=128, num_layers=2, batch_first=True, bidirectional=True)
        self.regression_layer = nn.Linear(128 * 2, 3)  # 128 * 2 because it's bidirectional
        
        
    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(input_values=input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        # Take the mean across the time dimension to get a fixed-size representation
        #pooled_output = hidden_states.mean(dim=1)
        #attention_output, _ = self.attention(hidden_states, hidden_states, hidden_states, key_padding_mask=~attention_mask.bool() if attention_mask is not None else None)

        # Average the attention output across the time dimension
        #pooled_output = attention_output.mean(dim=1)
        
        rnn_output, _ = self.rnn(hidden_states)

        # Average pooling over the time dimension
        pooled_output = rnn_output.mean(dim=1)
        return self.regression_layer(pooled_output)

In [4]:
def custom_collate(batch):
    input_values = [item['input_values'].squeeze(0) for item in batch]
    attention_mask = [item['attention_mask'].squeeze(0) if 'attention_mask' in item else torch.ones_like(item['input_values'].squeeze(0)) for item in batch]
    labels = torch.stack([item['labels'] for item in batch])

    # Pad input values and attention mask to the longest in the batch
    input_values_padded = pad_sequence(input_values, batch_first=True)
    attention_mask_padded = pad_sequence(attention_mask, batch_first=True)

    return {
        'input_values': input_values_padded,
        'attention_mask': attention_mask_padded,
        'labels': labels
    }

In [5]:
#checkpoint_path = "model_checkpoint.pth"
checkpoint_path = "model_checkpoint_sampled.pth"

In [6]:
def save_checkpoint(model, optimizer, epoch, filename=checkpoint_path):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch
    }
    torch.save(checkpoint, filename)
    print(f"Checkpoint saved at epoch {epoch + 1}")

def load_checkpoint(model, optimizer, filename=checkpoint_path):
    if os.path.isfile(filename):
        checkpoint = torch.load(filename, map_location="cpu", weights_only=True)  # Use "cpu" if using CPU, change if using GPU
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming training from epoch {start_epoch}")
        return start_epoch
    else:
        print("No checkpoint found, starting from scratch.")
        return 0


In [7]:
def train(model, train_dataloader, test_dataloader, epochs=5):
    optimizer = AdamW(model.parameters(), lr=1e-4)
    """if torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")"""

    model.train()

    for epoch in range(epochs):
        epoch_loss = 0
        optimizer.zero_grad()

        for batch in tqdm(train_dataloader):
            input_values = batch['input_values'].to("cpu")
            attention_mask = batch['attention_mask'].to("cpu")
            labels = batch['labels'].to("cpu")
            
            optimizer.zero_grad()
            outputs = model(input_values=input_values, attention_mask=attention_mask)
            loss = nn.MSELoss()(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_loss / len(train_dataloader)}")

        save_checkpoint(model, optimizer, epoch)

        # Validation Loop
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in test_dataloader:
                input_values = batch['input_values'].to("cpu")
                attention_mask = batch['attention_mask'].to("cpu")
                labels = batch['labels'].to("cpu")
                
                outputs = model(input_values=input_values, attention_mask=attention_mask)
                loss = nn.MSELoss()(outputs, labels)
                val_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Validation Loss: {val_loss / len(test_dataloader)}")


In [8]:
def load_trained_model(checkpoint_path):
    model = Wav2Vec2ForEmotionRegression().to("cpu")
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
    if os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(checkpoint['model_state_dict'])
        print("Loaded trained model from checkpoint.")
    else:
        print("Checkpoint not found. Using untrained model.")
    
    return model, processor

In [9]:
def predict_emotion(model, processor, wav_data):
    model.eval()
    inputs = processor(wav_data, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs['input_values'].to("cpu")
    attention_mask = inputs['attention_mask'].to("cpu") if 'attention_mask' in inputs else None

    with torch.no_grad():
        outputs = model(input_values=input_values, attention_mask=attention_mask)
    
    valence, arousal, dominance = outputs.squeeze().tolist()
    return {
        "Valence": valence,
        "Arousal": arousal,
        "Dominance": dominance
    }

In [10]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForEmotionRegression().to("cpu")

df = pd.read_pickle("../../data/IEMOCAP_useful")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/configuration_utils.py:302: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [15]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_sampled = df_shuffled.sample(frac=0.05, random_state=42)

df_sampled

,Turn_Name,Valence,Arousal,Dominance,wav_file
7360,Ses02F_script03_2_F043,0.3,1.0,0.8,"[-0.0095825195, -0.010650635, 0.00021362305, -..."
2184,Ses04M_script01_1_F039,0.5,0.6,0.6,"[-0.0015258789, -0.0014953613, -0.0017089844, ..."
8218,Ses03F_script03_1_M001,0.5,0.6,0.8,"[0.0010681152, 0.0010375977, 0.00079345703, 0...."
6160,Ses02F_impro01_F021,0.3,0.7,0.5,"[-0.01159668, -0.01473999, -0.015319824, -0.01..."
9082,Ses04M_impro02_M008,0.4,0.5,0.5,"[-0.0005187988, -0.00030517578, 0.0, 0.0002136..."
...,...,...,...,...,...
532,Ses03M_impro07_F015,0.7,0.8,0.6,"[6.1035156e-05, 0.00015258789, -0.0007019043, ..."
7363,Ses03M_script02_1_F014,0.3,0.9,0.9,"[0.0011291504, 0.001159668, 0.0010375977, 0.00..."
3428,Ses03M_script03_2_F015,0.5,0.5,0.6,"[-0.0007019043, -0.0014648438, -0.0010070801, ..."
9275,Ses03F_script03_1_F019,0.7,0.8,0.9,"[0.003112793, 0.004180908, 0.00390625, 0.00363..."


In [12]:
train_df, test_df = train_test_split(df_sampled, test_size=0.2, random_state=42)

# Create datasets
train_dataset = EmotionDataset(train_df, processor)
test_dataset = EmotionDataset(test_df, processor)

In [13]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=custom_collate)

In [14]:
train(model, train_dataloader, test_dataloader)


  0%|          | 0/101 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
100%|██████████| 101/101 [09:24<00:00,  5.59s/it]


Epoch 1/5, Training Loss: 0.038763353960717666
Checkpoint saved at epoch 1
Epoch 1/5, Validation Loss: 0.02870374910819989


100%|██████████| 101/101 [06:01<00:00,  3.58s/it]


Epoch 2/5, Training Loss: 0.026898189028263977
Checkpoint saved at epoch 2
Epoch 2/5, Validation Loss: 0.027851401828229427


100%|██████████| 101/101 [05:58<00:00,  3.55s/it]


Epoch 3/5, Training Loss: 0.026507534206577456
Checkpoint saved at epoch 3
Epoch 3/5, Validation Loss: 0.028703836186860617


100%|██████████| 101/101 [06:07<00:00,  3.64s/it]


Epoch 4/5, Training Loss: 0.025893836438950924
Checkpoint saved at epoch 4
Epoch 4/5, Validation Loss: 0.0280832380701143


100%|██████████| 101/101 [06:17<00:00,  3.73s/it]


Epoch 5/5, Training Loss: 0.025817957507344978
Checkpoint saved at epoch 5
Epoch 5/5, Validation Loss: 0.02829595970419737
